In [1]:
#import libraries
import pandas as pd


In [2]:
# import dataset
df=pd.read_csv('../data/raw/train.csv')
df.head()


,policy_id,policy_tenure,age_of_car,age_of_policyholder,area_cluster,population_density,make,segment,model,fuel_type,...,is_brake_assist,is_power_door_locks,is_central_locking,is_power_steering,is_driver_seat_height_adjustable,is_day_night_rear_view_mirror,is_ecw,is_speed_alert,ncap_rating,is_claim
0,ID00001,0.515874,0.05,0.644231,C1,4990,1,A,M1,CNG,...,No,No,No,Yes,No,No,No,Yes,0,0
1,ID00002,0.672619,0.02,0.375000,C2,27003,1,A,M1,CNG,...,No,No,No,Yes,No,No,No,Yes,0,0
2,ID00003,0.841110,0.02,0.384615,C3,4076,1,A,M1,CNG,...,No,No,No,Yes,No,No,No,Yes,0,0
3,ID00004,0.900277,0.11,0.432692,C4,21622,1,C1,M2,Petrol,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,2,0
4,ID00005,0.596403,0.11,0.634615,C5,34738,2,A,M3,Petrol,...,No,Yes,Yes,Yes,No,Yes,Yes,Yes,2,0


In [3]:
#1. seperate target and features
X=df.drop(columns=["is_claim"])
Y=df["is_claim"]
print("Features Shape",X.shape)
print("Target Shape",Y.shape)

Features Shape (58592, 43)
Target Shape (58592,)


In [4]:
#2. remove indentifiers
X= X.drop(columns=["policy_id"])
print(X.shape)

(58592, 42)


In [5]:
#3. Find invalid/suspicious values e.g age<0
X.describe().T[['min','max']]

,min,max
policy_tenure,0.002735,1.396641
age_of_car,0.000000,1.000000
age_of_policyholder,0.288462,1.000000
population_density,290.000000,73430.000000
make,1.000000,5.000000
airbags,1.000000,6.000000
displacement,796.000000,1498.000000
cylinder,3.000000,4.000000
gear_box,5.000000,6.000000
turning_radius,4.500000,5.200000


In [6]:
type_counts=X.apply(lambda col:col.apply(type).nunique())
type_counts


policy_tenure                       1
age_of_car                          1
age_of_policyholder                 1
area_cluster                        1
population_density                  1
make                                1
segment                             1
model                               1
fuel_type                           1
max_torque                          1
max_power                           1
engine_type                         1
airbags                             1
is_esc                              1
is_adjustable_steering              1
is_tpms                             1
is_parking_sensors                  1
is_parking_camera                   1
rear_brakes_type                    1
displacement                        1
cylinder                            1
transmission_type                   1
gear_box                            1
steering_type                       1
turning_radius                      1
length                              1
width       

# here we do not have any mixed data types inside cols.

In [7]:
vals=X[['max_torque','max_power']].apply(lambda val:val.unique())
vals

,max_torque,max_power
0,60Nm@3500rpm,40.36bhp@6000rpm
1,113Nm@4400rpm,88.50bhp@6000rpm
2,91Nm@4250rpm,67.06bhp@5500rpm
3,250Nm@2750rpm,113.45bhp@4000rpm
4,200Nm@3000rpm,88.77bhp@4000rpm
5,82.1Nm@3400rpm,55.92bhp@5300rpm
6,200Nm@1750rpm,97.89bhp@3600rpm
7,85Nm@3000rpm,61.68bhp@6000rpm
8,170Nm@4000rpm,118.36bhp@5500rpm


In [8]:
# x=X['max_torque'].str.contains('@') # also check for 'Nm' and 'rpm'
# x.all()
x=X['max_power'].str.contains('rpm') # also check for 'bhp' and '@'
x.all()

np.True_

#6. Feature extraction
 we will extract features from two cols 'max_torque' and 'max_power' and will make 4 additional features from them.

In [9]:
#splitting features
X[['torque_nm','torque_rpm']]= X['max_torque'].str.split('@',expand=True)
X[['power_bhp','power_rpm']]= X['max_power'].str.split('@',expand=True)


In [10]:
#removing units 
X['torque_nm']=X['torque_nm'].str.replace('Nm','')
X['torque_rpm']=X['torque_rpm'].str.replace('rpm','')
X['power_bhp']=X['power_bhp'].str.replace('bhp','')
X['power_rpm']=X['power_rpm'].str.replace('rpm','')

In [11]:
#changing str to numeric

X['torque_nm']=pd.to_numeric(X['torque_nm'])
X['torque_rpm']=pd.to_numeric(X['torque_rpm'])
X['power_bhp']=pd.to_numeric(X['power_bhp'])
X['power_rpm']=pd.to_numeric(X['power_rpm'])
X[['torque_nm', 'torque_rpm','power_bhp','power_rpm']].dtypes


torque_nm     float64
torque_rpm      int64
power_bhp     float64
power_rpm       int64
dtype: object

In [12]:
#remove cols
X=X.drop(columns=['max_torque','max_power'])


In [13]:
X[['torque_nm', 'torque_rpm', 'power_bhp', 'power_rpm']].head()

,torque_nm,torque_rpm,power_bhp,power_rpm
0,60.0,3500,40.36,6000
1,60.0,3500,40.36,6000
2,60.0,3500,40.36,6000
3,113.0,4400,88.50,6000
4,91.0,4250,67.06,5500



#Extracted Features
- torque_nm
- torque_rpm
- power_bhp
- power_rpm

#Observation

The structured text features were successfully transformed into numerical features suitable for machine learning. The resulting dataset contains 44 features after removing the original text columns.

#7.Inspect Categorical Features

In [14]:
#getting categorical cols
categorical_cols=X.select_dtypes(include='str').columns
categorical_cols

Index(['area_cluster', 'segment', 'model', 'fuel_type', 'engine_type',
       'is_esc', 'is_adjustable_steering', 'is_tpms', 'is_parking_sensors',
       'is_parking_camera', 'rear_brakes_type', 'transmission_type',
       'steering_type', 'is_front_fog_lights', 'is_rear_window_wiper',
       'is_rear_window_washer', 'is_rear_window_defogger', 'is_brake_assist',
       'is_power_door_locks', 'is_central_locking', 'is_power_steering',
       'is_driver_seat_height_adjustable', 'is_day_night_rear_view_mirror',
       'is_ecw', 'is_speed_alert'],
      dtype='str')

In [15]:
#check number of unique values for every col separately
X[categorical_cols].nunique()

area_cluster                        22
segment                              6
model                               11
fuel_type                            3
engine_type                         11
is_esc                               2
is_adjustable_steering               2
is_tpms                              2
is_parking_sensors                   2
is_parking_camera                    2
rear_brakes_type                     2
transmission_type                    2
steering_type                        3
is_front_fog_lights                  2
is_rear_window_wiper                 2
is_rear_window_washer                2
is_rear_window_defogger              2
is_brake_assist                      2
is_power_door_locks                  2
is_central_locking                   2
is_power_steering                    2
is_driver_seat_height_adjustable     2
is_day_night_rear_view_mirror        2
is_ecw                               2
is_speed_alert                       2
dtype: int64

In [16]:
#check unique values inside each categorical col
categories=X[categorical_cols].apply(lambda val:val.unique())
categories

area_cluster                        [C1, C2, C3, C4, C5, C6, C7, C8, C9, C10, C11,...
segment                                                  [A, C1, C2, B2, B1, Utility]
model                                  [M1, M2, M3, M4, M5, M6, M7, M8, M9, M10, M11]
fuel_type                                                       [CNG, Petrol, Diesel]
engine_type                         [F8D Petrol Engine, 1.2 L K12N Dualjet, 1.0 SC...
is_esc                                                                      [No, Yes]
is_adjustable_steering                                                      [No, Yes]
is_tpms                                                                     [No, Yes]
is_parking_sensors                                                          [Yes, No]
is_parking_camera                                                           [No, Yes]
rear_brakes_type                                                         [Drum, Disc]
transmission_type                                     

In [17]:
#check for whitespaces inside cols
whitespaces_check=X[categorical_cols].apply(lambda val:val.str.strip().ne(val)).sum()
whitespaces_check

area_cluster                        0
segment                             0
model                               0
fuel_type                           0
engine_type                         0
is_esc                              0
is_adjustable_steering              0
is_tpms                             0
is_parking_sensors                  0
is_parking_camera                   0
rear_brakes_type                    0
transmission_type                   0
steering_type                       0
is_front_fog_lights                 0
is_rear_window_wiper                0
is_rear_window_washer               0
is_rear_window_defogger             0
is_brake_assist                     0
is_power_door_locks                 0
is_central_locking                  0
is_power_steering                   0
is_driver_seat_height_adjustable    0
is_day_night_rear_view_mirror       0
is_ecw                              0
is_speed_alert                      0
dtype: int64

In [18]:
#check for inconsistent capitalization
X[categorical_cols].apply(lambda val:val.str.lower().nunique())==X[categorical_cols].nunique()

area_cluster                        True
segment                             True
model                               True
fuel_type                           True
engine_type                         True
is_esc                              True
is_adjustable_steering              True
is_tpms                             True
is_parking_sensors                  True
is_parking_camera                   True
rear_brakes_type                    True
transmission_type                   True
steering_type                       True
is_front_fog_lights                 True
is_rear_window_wiper                True
is_rear_window_washer               True
is_rear_window_defogger             True
is_brake_assist                     True
is_power_door_locks                 True
is_central_locking                  True
is_power_steering                   True
is_driver_seat_height_adjustable    True
is_day_night_rear_view_mirror       True
is_ecw                              True
is_speed_alert  

In [19]:
numerical_cols=X.select_dtypes(include=['int64','float64']).columns
numerical_cols

Index(['policy_tenure', 'age_of_car', 'age_of_policyholder',
       'population_density', 'make', 'airbags', 'displacement', 'cylinder',
       'gear_box', 'turning_radius', 'length', 'width', 'height',
       'gross_weight', 'ncap_rating', 'torque_nm', 'torque_rpm', 'power_bhp',
       'power_rpm'],
      dtype='str')

In [20]:
X[numerical_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
policy_tenure,58592.0,0.611246,0.414156,0.002735,0.210250,0.573792,1.039104,1.396641
age_of_car,58592.0,0.069424,0.056721,0.000000,0.020000,0.060000,0.110000,1.000000
age_of_policyholder,58592.0,0.469420,0.122886,0.288462,0.365385,0.451923,0.548077,1.000000
population_density,58592.0,18826.858667,17660.174792,290.000000,6112.000000,8794.000000,27003.000000,73430.000000
make,58592.0,1.763722,1.136988,1.000000,1.000000,1.000000,3.000000,5.000000
airbags,58592.0,3.137066,1.832641,1.000000,2.000000,2.000000,6.000000,6.000000
displacement,58592.0,1162.355851,266.304786,796.000000,796.000000,1197.000000,1493.000000,1498.000000
cylinder,58592.0,3.626963,0.483616,3.000000,3.000000,4.000000,4.000000,4.000000
gear_box,58592.0,5.245443,0.430353,5.000000,5.000000,5.000000,5.000000,6.000000
turning_radius,58592.0,4.852893,0.228061,4.500000,4.600000,4.800000,5.000000,5.200000


In [21]:
#IQR to find outliers
Q1=X[numerical_cols].quantile(0.25)
Q3=X[numerical_cols].quantile(0.75)
IQR=Q3-Q1
#lower bound
lower_bound=Q1-1.5*IQR
#upper bound
upper_bound=Q3+1.5*IQR



In [22]:
lower_bound

policy_tenure             -1.033032
age_of_car                -0.115000
age_of_policyholder        0.091346
population_density    -25224.500000
make                      -2.000000
airbags                   -4.000000
displacement            -249.500000
cylinder                   1.500000
gear_box                   5.000000
turning_radius             4.000000
length                  2620.000000
width                   1155.000000
height                  1235.000000
gross_weight             697.500000
ncap_rating               -4.500000
torque_nm               -150.000000
torque_rpm               275.000000
power_bhp                -45.935000
power_rpm               1000.000000
dtype: float64

In [23]:
upper_bound

policy_tenure              2.282386
age_of_car                 0.245000
age_of_policyholder        0.822115
population_density     58339.500000
make                       6.000000
airbags                   12.000000
displacement            2538.500000
cylinder                   5.500000
gear_box                   5.000000
turning_radius             5.600000
length                  4820.000000
width                   2115.000000
height                  1875.000000
gross_weight            1997.500000
ncap_rating                7.500000
torque_nm                410.000000
torque_rpm              6875.000000
power_bhp                184.185000
power_rpm               9000.000000
dtype: float64

In [24]:
#count outliers 
outlier_counts=((X[numerical_cols]< lower_bound)| (X[numerical_cols]> upper_bound)).sum()
outlier_counts

policy_tenure              0
age_of_car               269
age_of_policyholder      221
population_density      3647
make                       0
airbags                    0
displacement               0
cylinder                   0
gear_box               14381
turning_radius             0
length                     0
width                      0
height                     0
gross_weight               0
ncap_rating                0
torque_nm                  0
torque_rpm                 0
power_bhp                  0
power_rpm                  0
dtype: int64

In [26]:
X.loc[
    (X["population_density"] < lower_bound["population_density"]) |
    (X["population_density"] > upper_bound["population_density"]),
    "population_density"
].describe()

count     3647.000000
mean     72369.238826
std       2686.546086
min      65567.000000
25%      73430.000000
50%      73430.000000
75%      73430.000000
max      73430.000000
Name: population_density, dtype: float64

In [ ]:
X.loc[
    (X["age_of_car"] < lower_bound["age_of_car"]) |
    (X["age_of_car"] > upper_bound["age_of_car"]),
    "age_of_car"
].describe()

count    269.000000
mean       0.309257
std        0.098709
min        0.250000
25%        0.260000
50%        0.290000
75%        0.310000
max        1.000000
Name: age_of_car, dtype: float64

In [85]:
X.loc[
    (X["age_of_policyholder"] < lower_bound["age_of_policyholder"]) |
    (X["age_of_policyholder"] > upper_bound["age_of_policyholder"]),
    "age_of_policyholder"
].describe()

count    221.000000
mean       0.859554
std        0.036995
min        0.826923
25%        0.836538
50%        0.846154
75%        0.875000
max        1.000000
Name: age_of_policyholder, dtype: float64

## Cleaning Summary

The data cleaning process focused on identifying and addressing potential data quality issues before exploratory analysis and model development.

### Actions performed
- Removed `policy_id` because it is an identifier and does not provide useful predictive information.
- Checked numerical features for suspicious or impossible ranges.
- Verified that there were no true mixed Python data types within columns.
- Extracted numerical information from the structured `max_torque` and `max_power` features.
- Removed the original structured text columns after feature extraction.
- Inspected categorical features, their cardinality, whitespace, and capitalization consistency.
- Investigated numerical outliers using the IQR method.

### Outlier handling

The IQR method identified potential outliers in `age_of_car`, `age_of_policyholder`, `population_density`, and `gear_box`.

After inspection, these values were considered plausible and were retained. In particular, `gear_box` was flagged because its IQR was zero, while `population_density` contained a large concentration of observations at the maximum observed value.

Therefore, no observations were removed solely based on statistical outlier detection.

### Conclusion

The dataset is structurally clean enough to proceed to exploratory data analysis. Further transformations such as categorical encoding and numerical scaling will be handled later through a reusable machine learning preprocessing pipeline.